# Student Persistence Model
## 01. Data Exploration and Cleaning

This notebook explores the student information table from the Open University Learning Analytics Dataset (OULAD), checks data quality, and applies basic cleaning before building early-warning features.

## Setup

In [9]:
import pandas as pd

DATA_DIR = "data"

## Load student information

Each row is one student enrolled in one course (a student-course enrollment).

In [10]:
info = pd.read_csv(f"{DATA_DIR}/studentInfo.csv")
print(info.shape)
info.head()

(32593, 12)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


## Outcome distribution

In [11]:
info["final_result"].value_counts(normalize=True).round(3)

final_result
Pass           0.379
Withdrawn      0.312
Fail           0.216
Distinction    0.093
Name: proportion, dtype: float64

## Student groups

These groups will be used later to check whether the model performs equally well for different students.

In [12]:
for col in ["gender", "age_band", "highest_education", "imd_band", "disability"]:
    print(info[col].value_counts(), "\n")

gender
M    17875
F    14718
Name: count, dtype: int64 

age_band
0-35     22944
35-55     9433
55<=       216
Name: count, dtype: int64 

highest_education
A Level or Equivalent          14045
Lower Than A Level             13158
HE Qualification                4730
No Formal quals                  347
Post Graduate Qualification      313
Name: count, dtype: int64 

imd_band
20-30%     3654
30-40%     3539
10-20      3516
0-10%      3311
40-50%     3256
50-60%     3124
60-70%     2905
70-80%     2879
80-90%     2762
90-100%    2536
?          1111
Name: count, dtype: int64 

disability
N    29429
Y     3164
Name: count, dtype: int64 



## Data quality checks

### Repeat enrollments

Some students took more than one course. Each enrollment should still appear only once.

In [13]:
repeat_students = info["id_student"].duplicated().sum()
duplicate_enrollments = info.duplicated(["id_student", "code_module", "code_presentation"]).sum()

print(f"Rows belonging to students already seen: {repeat_students}")
print(f"Exact duplicate enrollments: {duplicate_enrollments}")

Rows belonging to students already seen: 3808
Exact duplicate enrollments: 0


### Missing values

`isna()` only finds empty cells. In this dataset, missing values in `imd_band` are recorded as `"?"`, so both are checked.

In [14]:
print("Empty cells:")
print(info.isna().sum()[info.isna().sum() > 0])

print("\nCells recorded as '?':")
print((info == "?").sum()[(info == "?").sum() > 0])

Empty cells:
Series([], dtype: int64)

Cells recorded as '?':
imd_band    1111
dtype: int64


## Cleaning

In [15]:
# Standardize the one imd_band label that is missing a % sign
info["imd_band"] = info["imd_band"].replace({"10-20": "10-20%"})

# Keep students with missing imd_band and label them explicitly
info["imd_band"] = info["imd_band"].replace({"?": "Unknown"})

info["imd_band"].value_counts()

imd_band
20-30%     3654
30-40%     3539
10-20%     3516
0-10%      3311
40-50%     3256
50-60%     3124
60-70%     2905
70-80%     2879
80-90%     2762
90-100%    2536
Unknown    1111
Name: count, dtype: int64

In [16]:
# Confirm the cleaning worked
assert (info == "?").sum().sum() == 0
assert "10-20" not in info["imd_band"].values
print("Cleaning checks passed.")

Cleaning checks passed.


## Summary

- **Records:** 32,593 student-course enrollments across 7 courses. 3,808 rows belong to students who took more than one course; there are no exact duplicate enrollments.
- **Outcome:** about 31% of enrollments ended in withdrawal. This is much higher than at most residential universities, so results may not transfer directly to that setting.
- **Small groups:** some groups are small (for example, 216 students aged 55+ and 313 with a postgraduate qualification), so results for them will be less certain.
- **Cleaning:** standardized one inconsistent `imd_band` label (`10-20` to `10-20%`) and relabeled 1,111 missing values (recorded as `?`) as `Unknown` instead of removing those students.